# R version of Proximity Analysis course

In [ ]:
library(here)
library(sf)
library(tidyverse)
library(leaflet)
library(leaflet.extras)


In [ ]:
releases = read_sf(here("data/toxic_release_pennsylvania/toxic_release_pennsylvania/toxic_release_pennsylvania.shp")) 
stations = read_sf(here("data/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations.shp"))

In [ ]:
print(st_crs(releases))
print(st_crs(stations))

In [ ]:
recent_release <- releases[361,]
recent_release

In [ ]:
distances <- st_distance(st_geometry(recent_release), st_geometry(stations))

In [ ]:
mean(distances)

In [ ]:
print(stations[which(distances == min(distances)),][c("ADDRESS", "LONGITUDE", "LATITUDE")])

In [ ]:
two_mile_buffer <- st_buffer(stations, 2*5280)
head(two_mile_buffer$geometry)

In [ ]:
disply_map <- function(m) {
  temp_html_file <- tempfile(fileext = ".html")
  htmlwidgets::saveWidget(m, temp_html_file, selfcontained = TRUE)
  browseURL(temp_html_file)
}

In [ ]:
m <- leaflet() %>% 
  addTiles() %>% 
  setView(lng = -75.1652, lat = 39.9526,  zoom = 11) %>% 
  addHeatmap(data = releases,
    lng = ~LONGITUDE, 
    lat = ~LATITUDE,
    radius = 15,
    blur = 15, 
    max = 1,
    minOpacity = 0.5
  ) %>% 
  addMarkers(data = stations,
    lng = ~LONGITUDE,
    lat = ~LATITUDE
  ) %>% 
  addPolygons(data = st_transform(two_mile_buffer, 4326))

disply_map(m)

In [ ]:
my_union <- st_combine(two_mile_buffer)
my_union

st_contains() ans similar return an integer vector of the geometries y contained in geometry X
To get true false values pass sparse = FALSE, which returns a matrix of true false values

In [ ]:
st_contains(my_union, st_geometry(releases[361,]))

In [ ]:
# The closest station is less than two miles away
st_contains(my_union, st_geometry(releases[361,]), sparse = FALSE)[1,1]

In [ ]:
# The closest station is more than two miles away
st_contains(my_union, st_geometry(releases[359,]), sparse = FALSE)[1,1]